# OpenCourses.AI Colab version

This notebook belongs to **Fundamentos de Transformers** and was published as a public Colab-ready artifact.

Colab is an external free execution option with variable resources. Run the setup cell before executing the rest of the notebook.


In [ ]:
# OpenCourses.AI Colab setup
import os, pathlib, subprocess, sys
REPO_URL = "https://github.com/opencourses-ai-colab/opencourse-fundamentos-de-transformers-d9d23ccb.git"
BRANCH = "main"
TARGET = pathlib.Path("/content/opencourses/opencourse-fundamentos-de-transformers-d9d23ccb")
TARGET.parent.mkdir(parents=True, exist_ok=True)
if not TARGET.exists():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, str(TARGET)], check=True)
os.environ['COURSE_ROOT'] = str(TARGET)
os.environ['ASSETS_DIR'] = str(TARGET / 'assets')
notebooks_dir = TARGET / 'notebooks'
if notebooks_dir.exists():
    os.chdir(notebooks_dir)
requirements = TARGET / 'requirements.txt'
if requirements.exists() and requirements.read_text(encoding='utf8').strip():
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(requirements)], check=True)
print(f'COURSE_ROOT={TARGET}')


<div style="border-bottom: 2px solid #1f2a44; padding-bottom: 14px; margin-bottom: 22px;">
  <div style="display: flex; align-items: center; justify-content: space-between; gap: 24px;">
    <div style="text-align: center; flex: 1; min-width: 260px;">
      <div style="font-size: 14px; letter-spacing: 0.04em; text-transform: uppercase; color: #5b6472;">Fundamentos de Transformers</div>
      <div style="font-size: 15px; font-weight: 700; color: #667085; margin-top: 6px;">Notebook 02</div>
      <div style="font-size: 26px; font-weight: 700; color: #1f2a44; margin-top: 2px;">Tokens, vocabulario y batches</div>
      <div style="font-size: 14px; color: #5b6472; margin-top: 8px;">Curso abierto</div>
    </div>
  </div>
</div>

<div style="display: flex; justify-content: space-between; gap: 16px; color: #3f4754; font-size: 14px; margin-bottom: 20px; flex-wrap: wrap;">
    <div><strong>Curso:</strong> Fundamentos de Transformers</div>
  <div><strong>Periodo:</strong> 2026-I</div>
</div>


## Presentación del notebook

<audio controls style="width: 100%; margin: 8px 0 18px 0;">
  <source src="../assets/audio/notebooks/02_tokens_vocabulario_y_batches_intro.mp3" type="audio/mpeg">
  Su navegador no puede reproducir el audio embebido.
</audio>


## Pregunta directriz

> ¿Cómo se transforma texto en índices y batches $X/Y$ sin confundir símbolos, etiquetas, posiciones y objetivos?

El notebook anterior formuló la predicción autoregresiva como una tarea local: dado un contexto, predecir el siguiente token. Ahora esa idea debe convertirse en datos entrenables. Para ello se necesita una cadena de representación precisa: texto, tokens, vocabulario, índices y ventanas desplazadas.

Esta etapa es fundacional porque un Transformer no recibe texto en lenguaje natural. Recibe tensores de enteros que identifican unidades discretas; si esa conversión no se entiende, embeddings, atención y pérdida quedan como operaciones desconectadas.

## Objetivos

Al finalizar este notebook, el estudiante debería poder:

1. Explicar la diferencia entre token, índice, posición y embedding dentro de la cadena de representación.
2. Construir un vocabulario bidireccional y usarlo para codificar y decodificar una secuencia corta.
3. Interpretar los índices como etiquetas enteras del vocabulario, no como coordenadas geométricas.
4. Crear batches autoregresivos $X$ y $Y$ con la misma forma tensorial pero con roles distintos.
5. Modificar `block_size` o el texto base y justificar cómo cambian longitud, vocabulario, ventanas disponibles y costo computacional.

## Marco conceptual

La tokenización no es una operación neutral. Decide qué unidades discretas podrá observar el modelo: caracteres, palabras, subpalabras o símbolos especiales. Esa decisión define el vocabulario $\mathcal{V}$, la longitud efectiva de las secuencias y la granularidad con la que se aprende el problema autoregresivo.

Un **token** es una unidad simbólica. Un **índice** es la etiqueta entera asignada a ese token dentro del vocabulario. Una **posición** indica el lugar que ocupa un token en la secuencia. Un **embedding**, que aparecerá en el siguiente notebook, será un vector entrenable asociado a un índice.

El curso inicia con tokenización por caracteres porque permite ver toda la cadena sin depender de un tokenizador externo. Los Transformers usados en producción suelen trabajar con subpalabras, por ejemplo mediante BPE u otros esquemas. Aquí esa complejidad se deja fuera para concentrarnos en el mecanismo autoregresivo y en la relación entre representación discreta y tensores.

La reversibilidad también importa. Si codificar convierte tokens en índices, decodificar debe permitir volver desde índices a tokens para inspeccionar batches, errores y generaciones. En esta etapa no buscamos semántica profunda; buscamos una representación consistente y auditable.

## Idea visual del proceso

<div style="text-align: center; margin: 12px 0 18px 0;">
  <img src="../assets/figures/02_tokens_vocabulario_y_batches.png" alt="Texto, tokens, vocabulario, índices y batches autoregresivos" style="width: min(980px, 98%); height: auto; border: 1px solid #d0d7de; border-radius: 6px;">
</div>

La figura se lee de izquierda a derecha: el texto se segmenta en tokens, cada token se consulta en un vocabulario y se reemplaza por una etiqueta entera. Con esa secuencia de índices se construyen ventanas alineadas `X/Y` para entrenamiento autoregresivo.

La parte crítica es no mezclar objetos: el token pertenece al lenguaje simbólico, el índice pertenece al vocabulario, la posición pertenece al orden de la secuencia y el batch pertenece a la organización computacional del entrenamiento.

## Formulación matemática

Para poder entrenar un modelo, primero se define una tokenización $\tau$ que transforma un texto en una secuencia de unidades discretas:

$$
\tau(\text{texto}) = (z_1, z_2, \ldots, z_T), \qquad z_t \in \mathcal{V}.
$$

Esta ecuación se lee así: el texto original se convierte en una secuencia ordenada de $T$ tokens, y cada token pertenece a un vocabulario finito $\mathcal{V}$.

Luego se define una función de indexación $\iota$ que asigna una etiqueta entera a cada token del vocabulario:

$$
\iota: \mathcal{V} \rightarrow \{0,1,\ldots,|\mathcal{V}|-1\}.
$$

Esta ecuación se lee así: cada token válido tiene un identificador entero único. Ese entero sirve para consultar tablas y construir tensores, pero no representa una magnitud semántica.

La secuencia codificada queda como:

$$
x_{1:T} = (\iota(z_1), \iota(z_2), \ldots, \iota(z_T)).
$$

Esta ecuación se lee así: cada token de la secuencia fue reemplazado por el índice que le corresponde en el vocabulario.

Para entrenamiento autoregresivo se usan ventanas desplazadas:

$$
X = x_{i:i+L}, \qquad Y = x_{i+1:i+L+1}.
$$

Esta ecuación se lee así: $X$ contiene una ventana de $L$ índices y $Y$ contiene la misma región del corpus corrida una posición hacia adelante. En cada columna, el modelo mira el índice de $X$ y debe asignar alta probabilidad al índice alineado en $Y$.

Advertencia importante: el índice $7$ no está geométricamente más cerca de $8$ que de $2$. Esa geometría aparece cuando los índices seleccionan vectores de embedding, no antes.


## Traducción tensorial

Una secuencia codificada tiene forma $(T,)$: una dimensión temporal con índices enteros. Un batch de entradas tiene forma $(B,L)$, donde $B$ es el número de ventanas y $L$ es la longitud de contexto usada en cada ventana.

$X$ y $Y$ tienen la misma forma porque cada posición de entrada necesita un objetivo alineado. La igualdad de forma no implica igualdad de significado: $X$ organiza contexto de entrada y $Y$ organiza los targets del siguiente token.

## Preparación del entorno

In [1]:
from pathlib import Path
import importlib
import math
import sys
import warnings

warnings.filterwarnings("ignore", message="Pandas requires version.*")

from IPython.display import display
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

import fundamentos_transformers.visualizacion as visualizacion

visualizacion = importlib.reload(visualizacion)
configurar_matplotlib = visualizacion.configurar_matplotlib
mostrar_tablas_en_columnas = visualizacion.mostrar_tablas_en_columnas

configurar_matplotlib()
torch.manual_seed(7)
np.random.seed(7)

## Experimento: de texto a batches autoregresivos

Usaremos un corpus miniatura para inspeccionar toda la cadena. La primera parte fija una construcción estática: texto, tokens, vocabulario, índices y formas de $X/Y$. La segunda parte usa un explorador interactivo para mover `start` y `block_size` y observar cómo cambia la ventana autoregresiva.

La intención no es crear un dataset realista, sino volver manipulable la relación entre posición en el corpus, índice de vocabulario, entrada $X$ y objetivo $Y$.

In [2]:
from fundamentos_transformers.tokenizacion import tokenizar_caracteres, construir_vocabulario, codificar, decodificar
from fundamentos_transformers.datos import crear_batch_autoregresivo

corpus = "atencion transforma tokens en contexto. atencion mezcla informacion."
tokens = tokenizar_caracteres(corpus)
token_a_id, id_a_token = construir_vocabulario(tokens)
ids = torch.tensor(codificar(tokens, token_a_id), dtype=torch.long)
block_size = 12
batch_size = 4
x, y = crear_batch_autoregresivo(ids, block_size=block_size, batch_size=batch_size, seed=3)

resumen_df = pd.DataFrame([
    ["caracteres del corpus", len(corpus), "Longitud textual original."],
    ["tokens", len(tokens), "Con tokenización por caracteres coincide con la longitud del texto."],
    ["vocabulario", len(token_a_id), "Número de símbolos distintos observados."],
    ["block_size", block_size, "Longitud L de cada ventana de entrada."],
    ["batch_size", batch_size, "Número de ventanas muestreadas."],
    ["forma de X", tuple(x.shape), "Contextos de entrada."],
    ["forma de Y", tuple(y.shape), "Targets desplazados una posición."],
], columns=["objeto", "valor", "interpretación"])
vocab_df = pd.DataFrame([
    {"índice": idx, "token": "<espacio>" if token == " " else token, "representación": repr(token)}
    for token, idx in sorted(token_a_id.items(), key=lambda kv: kv[1])
])
mostrar_tablas_en_columnas([
    ("Resumen del corpus", resumen_df),
    ("Vocabulario", vocab_df),
], min_width="320px")

objeto,valor,interpretación
caracteres del corpus,68,Longitud textual original.
tokens,68,Con tokenización por caracteres coincide con la longitud del texto.
vocabulario,17,Número de símbolos distintos observados.
block_size,12,Longitud L de cada ventana de entrada.
batch_size,4,Número de ventanas muestreadas.
forma de X,"(4, 12)",Contextos de entrada.
forma de Y,"(4, 12)",Targets desplazados una posición.
índice,token,representación
0,<espacio>,' '
1,.,'.'


La tabla de resumen fija las magnitudes principales. La tabla de vocabulario muestra que el índice es una entrada de una tabla, no una posición temporal del texto.

Ahora inspeccionaremos una fila del batch y después pasaremos a un explorador interactivo donde `start` y `block_size` se pueden modificar directamente.

In [3]:
fila = 0
alineacion_df = pd.DataFrame({
    "columna": list(range(block_size)),
    "posición corpus X": list(range(block_size)),
    "X_id": x[fila].tolist(),
    "X_token": ["<espacio>" if id_a_token[int(i)] == " " else id_a_token[int(i)] for i in x[fila]],
    "posición corpus Y": list(range(1, block_size + 1)),
    "Y_id": y[fila].tolist(),
    "Y_token_siguiente": ["<espacio>" if id_a_token[int(i)] == " " else id_a_token[int(i)] for i in y[fila]],
})
comparacion_block_size = []
for L in [4, 8, 12, 20]:
    ventanas_posibles = max(0, len(ids) - L)
    comparacion_block_size.append({
        "block_size": L,
        "ventanas posibles": ventanas_posibles,
        "forma esperada de una fila X": (L,),
        "lectura": "más contexto por ejemplo" if L > block_size else "menos contexto por ejemplo",
    })
comparacion_block_size_df = pd.DataFrame(comparacion_block_size)
mostrar_tablas_en_columnas([
    ("Alineación posición por posición", alineacion_df),
    ("Efecto de block_size", comparacion_block_size_df),
], min_width="420px")

columna,posición corpus X,X_id,X_token,posición corpus Y,Y_id,Y_token_siguiente
0,0,0,<espacio>,1,4,e
1,1,4,e,2,10,n
2,2,10,n,3,0,<espacio>
3,3,0,<espacio>,4,3,c
4,4,3,c,5,11,o
5,5,11,o,6,10,n
6,6,10,n,7,14,t
7,7,14,t,8,4,e
8,8,4,e,9,15,x
9,9,15,x,10,14,t


## Explorador interactivo de ventanas $X/Y$

En el siguiente componente, `start` controla la posición inicial de la ventana en el corpus y `block_size` controla cuántos tokens entran en $X$. Al moverlos, observe cuatro objetos distintos: posición temporal, token, índice y rol dentro de $X$ o $Y$.

In [4]:
import importlib
import fundamentos_transformers.componentes_interactivos as componentes_interactivos

componentes_interactivos = importlib.reload(componentes_interactivos)

explorador_batches = componentes_interactivos.crear_explorador_batches_autoregresivos(
    block_size_inicial=8,
    start_inicial=3,
)
display(explorador_batches)

## Interpretación

El desplazamiento entre $X$ y $Y$ parece una operación menor, pero convierte un corpus lineal en ejemplos supervisados de predicción del siguiente token. $X$ y $Y$ no son dos textos independientes: son dos vistas alineadas de la misma secuencia, separadas por una posición.

En cada columna, $X$ aporta el token visible y $Y$ aporta el token observado inmediatamente después. Esa pareja define el entrenamiento autoregresivo: dado lo que está disponible hasta una posición, el modelo debe poner probabilidad sobre el siguiente índice del vocabulario.

El explorador permite separar cuatro nociones que suelen confundirse. La **posición** indica dónde está un símbolo en el corpus. El **token** es la unidad simbólica observada. El **índice** es la etiqueta que ese token ocupa en el vocabulario. El **rol** indica si esa aparición funciona como entrada en $X$ o como objetivo en $Y$. El mismo índice puede aparecer en muchas posiciones distintas, y una misma posición puede ser objetivo de una columna y entrada de la siguiente.

Cambiar `block_size` modifica la longitud del contexto disponible por cada ejemplo. Un contexto mayor puede permitir dependencias más largas, pero también aumenta costo y reduce el número de ventanas distintas que pueden extraerse de un corpus corto.

La codificación es reversible dentro del vocabulario construido: podemos pasar de token a índice y de índice a token. Si aparece un token fuera del vocabulario, este esquema mínimo no sabe representarlo; los tokenizadores reales introducen mecanismos para manejar ese problema.


## Verificación de aprendizaje

Use el explorador interactivo y modifique una sola variable: `start`, `block_size` o el texto seleccionado. Luego reporte evidencia concreta:

1. el texto seleccionado y el tamaño de vocabulario $|\mathcal{V}|$;
2. el valor de `start` y `block_size` usados;
3. una fila alineada de $X$ y $Y$ con posición, token e índice;
4. una explicación de por qué índice y posición no son lo mismo;
5. una interpretación de cómo cambió el número de ventanas posibles al modificar `block_size`.

## Análisis crítico

Este notebook no enseña todavía a representar significado; enseña a construir el soporte discreto que hará posible el entrenamiento. La tokenización por caracteres es transparente y fácil de auditar, pero produce secuencias más largas que una tokenización por subpalabras.

El vocabulario observado depende del corpus. Por eso la codificación no debe entenderse como verdad universal sobre el lenguaje, sino como una convención operativa para un conjunto de datos y una tarea.

## Síntesis

$$
\text{texto} \rightarrow \text{tokens} \rightarrow \mathcal{V} \rightarrow x_{1:T} \rightarrow (X,Y)
$$

Esta cadena se lee así: el texto se segmenta, los tokens definen un vocabulario, el vocabulario permite asignar índices y esos índices se organizan en ventanas desplazadas para entrenar predicción del siguiente token.

<div style="border-left: 4px solid #1f2a44; padding: 10px 14px; background: #f6f8fa; margin: 14px 0;">
La idea central es distinguir objetos: token, índice, posición, contexto y objetivo. Si esos objetos se confunden, el resto del Transformer se vuelve opaco.
</div>


## Preguntas de discusión

1. ¿Qué gana y qué pierde una tokenización por caracteres frente a una por subpalabras?
2. ¿Por qué $X$ y $Y$ tienen la misma forma aunque representan roles distintos?
3. ¿Qué significa que un índice sea una etiqueta y no una coordenada?
4. ¿Cómo cambia el costo de entrenamiento si aumenta `block_size`?
5. ¿Qué problema aparece si el texto contiene un token que no existe en el vocabulario construido?

## Continuidad

Con índices listos, el siguiente paso es convertir etiquetas discretas en vectores entrenables. Esa conversión introduce la dimensión de canales $C$ y permite que el modelo procese representaciones continuas en lugar de enteros aislados.